# 3장 예제 — 3.12 Feature Engineering — 도메인 지식을 변수로 빚다

- **교재 출처**: manuscript/31_ch3_eda_fe.md
- **실행 방법**: example 폴더에서  python "3장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 09_3-9_단변량_시각화.py, 10_3-10_이변량·다변량_시각화.py, 11_3-11_상관분석.py
- **실행 상태**: 단독 실행 확인됨(선행 절 준비 코드 포함).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import pandas as pd
import numpy as np

### 📌 [선행 절 준비 코드 — 단독 실행용] 아래는 3.7~3.11절에서 만든 객체를 이 파일만으로 재현한 것이다. 교재 본문에서는 앞 절에서 이미 만들어져 있으므로 다시 실행할 필요가 없다.

> 앞 절에서 만든 변수/객체를 재현하는 코드입니다.

In [ ]:
def clean_data(path):
    """배터리 공정 데이터 정제 파이프라인"""
    df = pd.read_csv(path)
    # 1) 에러 코드 -> NaN
    df['건조로_1구간_온도'] = df['건조로_1구간_온도'].replace(9999, np.nan)
    # 2) 물리 범위 위반 -> NaN
    rules = {'믹싱_온도': (10, 50), '건조로_1구간_온도': (80, 160),
             '프레스_압력': (20, 80)}
    for col, (lo, hi) in rules.items():
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
    # 3) 결측치 -> 중앙값 대체
    num_cols = df.select_dtypes('number').columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    # 4) 중복 제거
    df = df.drop_duplicates(subset='Lot_ID')
    return df

df = clean_data('data/battery_process_data.csv')

num_cols = df.select_dtypes('number').columns.drop('불량_여부')

corr = df[num_cols].corr()
# ==================== [선행 절 준비 코드 끝] ==========================

### 파생변수 — 비율, 차이, 구간화

In [ ]:
# 비율: 단위 Gap당 프레스 압력 — 압연 강도의 대리 지표
df['압력_Gap_비율'] = df['프레스_압력'] / df['프레스_Gap']

# 차이: 건조로와 믹싱 공정의 온도 차 — 공정 간 열 이력 변화
df['온도_차이'] = df['건조로_1구간_온도'] - df['믹싱_온도']

In [ ]:
df['Gap_구간'] = pd.cut(df['프레스_Gap'], bins=[0, 90, 100, 999],
                       labels=['얇음', '정상', '두꺼움'])
print(df['Gap_구간'].value_counts())
print(df.groupby('Gap_구간', observed=True)['불량_여부'].mean().round(4))

### 범주 인코딩 — 원-핫

In [ ]:
dummies = pd.get_dummies(df['Gap_구간'], prefix='Gap', dtype=int)
print(pd.concat([df['Gap_구간'], dummies], axis=1).head(3))

# 교재 실행 결과 ------------------------------------------------------
#     Gap_구간  Gap_얇음  Gap_정상  Gap_두꺼움
#   0     정상       0       1        0
#   1     정상       0       1        0
#   2     정상       0       1        0

### 상호작용항

In [ ]:
df['RPMxGap'] = df['믹싱_RPM'] * df['프레스_Gap']
print(df['RPMxGap'].corr(df['불량_여부']).round(3))   # 0.226

### 도메인 기반 파생 — RPM 관리범위 이탈 플래그의 실검증

In [ ]:
df['RPM_이탈'] = ((df['믹싱_RPM'] < 1600) | (df['믹싱_RPM'] > 1900)).astype(int)

print(df['RPM_이탈'].value_counts())
print(pd.crosstab(df['RPM_이탈'], df['불량_여부']))
print(df.groupby('RPM_이탈')['불량_여부'].mean().round(4))

### 특징 선택 개요 — 분산 필터와 상관 필터

In [ ]:
high = corr.abs().where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print(high[high > 0.8].round(3))